<table border=0 width="100%"><tr><td><p align="left"><img src="..\img\logo.png" align="left" width=300></p></td><td><font size=3><B>一个端到端的项目：房价预测 (郑海超)</B></font></td></tr></table>

# 一个端到端的项目：房价预测
- 这个案例很成熟，包括数据准备、模型训练、模型评估等
- 大家可以通过这个案例来学习线性回归的整个流程


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.model_selection import train_test_split
from pandas.plotting import scatter_matrix
%matplotlib inline

import pandas as pd
import numpy as np
import jieba
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline

reference - Aurélien Géron. Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow. 2nd Edition.

## 导入数据
California housing prices

In [ ]:
housing = pd.read_csv("housing.csv")

## 数据可视化与探索

In [ ]:
housing_copy = housing.copy()

In [ ]:
housing_copy.head()

In [ ]:
housing_copy.plot(kind="scatter", x="longitude", y="latitude",
                  s=housing["population"]/100,
                  c="median_house_value",
                  alpha = 0.4,
                  cmap=plt.get_cmap("jet"),  # Colormap, you could try gray
                  colorbar=True)
plt.savefig("housing price.png")

In [ ]:
if 'ocean_proximity' in housing_copy.columns:
    # housing_copy = housing_copy.copy()
    housing_copy['ocean_proximity'] = pd.factorize(housing_copy['ocean_proximity'])[0]

In [ ]:
housing_copy["ocean_proximity"].value_counts()


In [ ]:
corr_matrix = housing_copy.corr()

In [ ]:
corr_matrix.median_house_value.sort_values()

In [ ]:
corr_matrix.corr()["median_house_value"].sort_values(ascending = False)

## 生成一些特征：create some features

In [ ]:
# 我们可以生成一些你认为有用的特征吗？
housing_copy["rooms_per_household"] = housing_copy.total_rooms / housing_copy.households
housing_copy["bedrooms_per_room"] = housing_copy.total_bedrooms / housing_copy.total_rooms
housing_copy["population_per_household"] = housing_copy.population / housing_copy.households

In [ ]:
housing_copy.corr()["median_house_value"].sort_values(ascending = False)

## 机器学习数据准备

### fill missing value

In [ ]:
housing.isnull().any()

In [ ]:
# fill the nan values
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')

In [ ]:
# 去掉距离海洋的距离，这个是分类数据，用imputer均值替换缺失值是不合适的，因此单独把ocean_proximity去掉，保留数值数据
housing_num = housing_copy.drop("ocean_proximity", axis=1)

In [ ]:
imputer.fit(housing_num)
X = imputer.transform(housing_num)

In [ ]:
housing_tr = pd.DataFrame(X, columns=housing_num.columns)

In [ ]:
housing_cat = housing["ocean_proximity"]
housing_cat

###  LabelBinarizer

In [ ]:
housing_cat

In [ ]:
from sklearn.preprocessing import LabelBinarizer

In [ ]:
encoder = LabelBinarizer()

In [ ]:
housing_cat_1hot = encoder.fit_transform(housing_cat)

In [ ]:
housing_cat_1hot

In [ ]:
encoder.classes_

### StandardScaler

In [ ]:
# StandardScaler
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
housing_num_scale = scaler.fit_transform(X)

In [ ]:
housing_num_scale.shape

In [ ]:
housing_prepared = np.column_stack([housing_num_scale,housing_cat_1hot])

## 模型训练与评估

### split the dataset

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelBinarizer
import math

In [ ]:
# use Pipeline to prepare the data for ML
def prepareData(housing):
    housing_labels = housing_copy["median_house_value"].copy()
    housing_labels = housing_labels.apply(lambda x: math.log(x))
    
    housing = housing_copy.drop(["median_house_value"], axis=1)
    housing_num = housing_copy.drop("ocean_proximity", axis=1)
    num_pipeline = Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
         ("std_scaler",StandardScaler())
    ])
    housing_num_tr = num_pipeline.fit_transform(housing_num)
    
    encoder = LabelBinarizer()
    housing_cat = housing_copy["ocean_proximity"]
    housing_cat_1hot = encoder.fit_transform(housing_cat)
    
    housing_prepared = np.column_stack([housing_num_tr,housing_cat_1hot])
    
    return housing_prepared,housing_labels

In [ ]:
housing = pd.read_csv("housing.csv")
housing_prepared,housing_labels = prepareData(housing)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(housing_prepared, housing_labels,test_size = 0.3,random_state=42)

### LinearRegression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [ ]:
lin_reg = LinearRegression()

In [ ]:
scores = cross_val_score(lin_reg, 
                         X_train, y_train,
                         scoring="neg_mean_squared_error", 
                         cv=5)

In [ ]:
scores

cross_val_score用于执行交叉验证的函数。它的运行过程可以分为以下几个步骤：

将数据集划分为 k 个折（folds）：首先，cross_val_score 函数将输入的数据集划分为 k 个相等大小的折。这个过程称为 k 折交叉验证（k-fold cross-validation）。每个折都包含大致相等数量的样本，并且保持了原始数据集中样本的分布。

- 对每个折进行迭代：接下来，cross_val_score 函数对每个折进行迭代，并进行以下步骤：

> - a. 将当前折作为测试集：选择当前折作为测试集，其他 k-1 个折作为训练集。

> - b. 在训练集上拟合模型：使用训练集的数据和标签，根据指定的模型或估计器，拟合一个模型。

> - c. 在测试集上进行预测：使用拟合好的模型对当前折的测试集进行预测。

> - d. 计算评估指标：根据预测结果和测试集的真实标签，计算所选择的评估指标的值。

- 收集评估指标的值：对于每个折，cross_val_score 函数都会计算评估指标的值。然后，它会收集这些值，并返回一个包含这些值的数组。

- 返回结果：cross_val_score 函数最终返回一个包含每个折的评估指标值的数组。可以根据需要使用这些值进行模型选择、性能比较等操作。

在使用 cross_val_score 函数进行交叉验证时，通过设置 scoring="neg_mean_squared_error" 参数，将评估指标设定为负均方误差（Negative Mean Squared Error）。

负均方误差是一种常用的回归模型评估指标，它衡量了模型预测值与真实值之间的平均平方误差。通过取负值，可以将评估指标转化为一个需要最大化的值，而不是需要最小化的值。这样做的原因主要有两个方面：

- 一致性： 通过将评估指标设定为负值，可以使得评估指标越大越好，与其他评估指标的一致性保持一致。通常情况下，评估指标越大表示模型的性能越好。因此，将负均方误差作为评估指标可以与其他指标（如准确率、R方等）保持一致。

- 便于比较： 通过取负值，可以将评估指标的方向与其他评估指标保持一致，从而使得不同模型的性能比较更加直观。比如，在交叉验证中使用 cross_val_score 函数评估多个模型时，可以直接比较评估指标的数值大小，而不需要关注具体的正负符号。

需要注意的是，在使用负均方误差作为评估指标时，评估指标的数值越大表示模型的性能越好。因此，在使用 cross_val_score 函数进行交叉验证时，可以选择最大化评估指标的数值来选择最佳模型。

In [ ]:
scores = -scores
scores.mean()

In [ ]:
lin_reg.fit(X_train, y_train)

### evaluate the model using test dataset

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
y_test_predicted = lin_reg.predict(X_test)
mean_squared_error(y_pred=y_test_predicted,y_true=y_test)